26688 PDFs

     ↓
Extract Text

     ↓
case_name

year
legal_text

     ↓
clean_cases.parquet

In [3]:
import pandas as pd
import fitz

from pathlib import Path
from tqdm import tqdm

In [4]:
DATA_PATH = Path(
    "../data/raw/supreme_court_judgments"
)

pdf_files = sorted(
    list(DATA_PATH.rglob("*.pdf"))
)

print(len(pdf_files))

26688


In [5]:
def extract_text_pymupdf(pdf_path):

    text = ""

    try:

        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text()

        doc.close()

    except Exception:
        pass

    return text

In [6]:
sample_text = extract_text_pymupdf(
    pdf_files[0]
)

print(sample_text[:1000])

A.K. Gopalan vs The State Of Madras.Union Of India: ... on 19
May, 1950
Equivalent citations: 1950 AIR 27, 1950 SCR 88, AIR 1950 SUPREME COURT
27, 1963 MADLW 638
Author: Hiralal J. Kania
Bench: Hiralal J. Kania, Saiyid Fazal Ali, Mehr Chand Mahajan, B.K.
Mukherjea
           PETITIONER:
A.K. GOPALAN
        Vs.
RESPONDENT:
THE STATE OF MADRAS.UNION OF INDIA: INTERVENER.
DATE OF JUDGMENT:
19/05/1950
BENCH:
KANIA, HIRALAL J. (CJ)
BENCH:
KANIA, HIRALAL J. (CJ)
FAZAL ALI, SAIYID
SASTRI, M. PATANJALI
MAHAJAN, MEHR CHAND
DAS, SUDHI RANJAN
MUKHERJEA, B.K.
CITATION:
 1950 AIR   27            1950 SCR   88
 CITATOR INFO :
 F          1951 SC 157  (21)
 F          1951 SC 270  (5,6)
 F          1951 SC 301  (10)
 F          1951 SC 332  (344)
 E          1952 SC  75  (45)
 RF         1952 SC 123  (6)
 F          1952 SC 181  (6,27,29,33)
 D          1952 SC 196  (16)
 RF         1952 SC 252  (106)
 R          1952 SC 366  (16)
 E&F        1952 SC 369  (90,93)
 F          1953 SC 451  (7)
 E&F   

Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950
Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950
Equivalent citations: 1950 AIR 15, 1950 SCR 30, AIR 1950 SUPREME COURT
15
Bench: Hiralal J. Kania, Saiyid Fazal Ali, Mehr Chand Mahajan
PETITIONER:
ABDULLA AHMED
Vs.
RESPONDENT:
ANIMENDRA KISSEN MITTER.
DATE OF JUDGMENT:
14/03/1950
BENCH:
SASTRI, M. PATANJALI
BENCH:
SASTRI, M. PATANJALI
DAS, SUDHI RANJAN
KANIA, HIRALAL J. (CJ)
FAZAL ALI, SAIYID
MAHAJAN, MEHR CHAND
CITATION:
1950 AIR 15 1950 SCR 30
CITATOR INFO :
F 1975 SC 32 (19)
E 1980 SC 17 (36)
E 1990 SC1833 (17)
ACT:
Contract Agency Estate broker--Authority to negotiate a
sale' and'secure purchaser '--Whether empowers broker to
conclude contract--Construction of contract--Broker finding
ready and willing to buy for price fixed by principal con-
cluding contract with same purchase for lower price -Bro-
ker's right to commission -power of agents.
principal--Principal
HEADNOTE:
The appellant, an estate broker, was empl

In [7]:
from pathlib import Path

CHUNK_DIR = Path(
    "../data/processed/chunks"
)

CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [8]:
import re

def clean_legal_text(text):

    text = re.sub(
        r'Indian Kanoon - http.*?\d+',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()

In [9]:
CHUNK_SIZE = 1000

for start_idx in range(
    0,
    len(pdf_files),
    CHUNK_SIZE
):

    chunk_number = start_idx // CHUNK_SIZE

    chunk_file = (
        CHUNK_DIR /
        f"chunk_{chunk_number:03d}.parquet"
    )

    # already processed
    if chunk_file.exists():

        print(
            f"Skipping Chunk {chunk_number}"
        )

        continue

    records = []

    end_idx = min(
        start_idx + CHUNK_SIZE,
        len(pdf_files)
    )

    current_files = pdf_files[
        start_idx:end_idx
    ]

    for pdf_path in tqdm(
        current_files,
        desc=f"Chunk {chunk_number}"
    ):

        try:

            text = extract_text_pymupdf(
                pdf_path
            )

            text = clean_legal_text(
                text
            )

            records.append({

                "case_name":
                pdf_path.stem,

                "year":
                pdf_path.parent.name,

                "legal_text":
                text

            })

        except Exception:
            pass

    pd.DataFrame(
        records
    ).to_parquet(
        chunk_file,
        index=False
    )

    print(
        f"Saved {chunk_file}"
    )

Chunk 0:   0%|          | 0/1000 [00:00<?, ?it/s]

Chunk 0: 100%|██████████| 1000/1000 [01:04<00:00, 15.57it/s]


Saved ..\data\processed\chunks\chunk_000.parquet


Chunk 1: 100%|██████████| 1000/1000 [01:02<00:00, 15.99it/s]


Saved ..\data\processed\chunks\chunk_001.parquet


...

Saved ..\data\processed\chunks\chunk_024.parquet


Chunk 25: 100%|██████████| 1000/1000 [01:19<00:00, 12.52it/s]


Saved ..\data\processed\chunks\chunk_025.parquet


Chunk 26: 100%|██████████| 688/688 [00:57<00:00, 12.02it/s]


Saved ..\data\processed\chunks\chunk_026.parquet


In [2]:
from pathlib import Path

chunk_files = sorted(
    Path("../data/processed/chunks")
    .glob("*.parquet")
)

print("Total Chunks:", len(chunk_files))

import pandas as pd

total_rows = 0

for file in chunk_files:
    df = pd.read_parquet(file)
    total_rows += len(df)

print("Total Cases:", total_rows)

Total Chunks: 27
Total Cases: 26688


In [3]:
import pandas as pd
from pathlib import Path

chunk_files = sorted(
    Path("../data/processed/chunks")
    .glob("*.parquet")
)

dfs = []

for file in chunk_files:
    dfs.append(
        pd.read_parquet(file)
    )

final_df = pd.concat(
    dfs,
    ignore_index=True
)

print(final_df.shape)

(26688, 3)


In [4]:
final_df.to_parquet(
    "../data/processed/clean_cases.parquet",
    index=False
)

print("Saved")

Saved


In [5]:
print(final_df.head())

print(
    final_df["legal_text"]
    .isna()
    .sum()
)

print(
    (final_df["legal_text"]
     .str.len()==0)
    .sum()
)

                                           case_name  year  \
0  A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_In...  1950   
1  A_M_Mair_Co_vs_Gordhandass_Sagarmull_on_30_Nov...  1950   
2  Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14...  1950   
3  Arjuna_Lal_Misra_vs_The_State_on_30_November_1...  1950   
4  Ashutosh_Lahiry_vs_The_State_Of_Delhi_And_Anr_...  1950   

                                          legal_text  
0  A.K. Gopalan vs The State Of Madras.Union Of I...  
1  A.M. Mair & Co vs Gordhandass Sagarmull on 30 ...  
2  Abdulla Ahmed vs Animendra Kissen Mitter on 14...  
3  Arjuna Lal Misra vs The State on 30 November, ...  
4  Ashutosh Lahiry vs The State Of Delhi And Anr....  
0
0
